In [ ]:
# Setup for Kaggle or Google Colab environments
# Run this cell to install any missing dependencies!
!pip install -q pandas numpy scikit-learn scipy matplotlib seaborn mplsoccer tqdm

# Task 6 - Evaluation, Validation & Reporting


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Setup environment and constants


In [ ]:
# -----------------------------------------------------------
# Data Loading for Evaluation
# -----------------------------------------------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from scipy.stats import spearmanr

# Set plot style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)

# Load raw dataset (for features and hidden cols)
try:
    df_raw = pd.read_csv("../../data/extract-feature/extract_feature_worldcup_2018_2022_raw.csv")
    labels_scaled = pd.read_csv("labels_scaled.csv")
    labels_unscaled = pd.read_csv("labels_unscaled.csv")
except FileNotFoundError:
    df_raw = pd.read_csv("shots_worldcup_2018_2022_raw.csv")
    labels_scaled = pd.read_csv("labels_scaled.csv")
    labels_unscaled = pd.read_csv("labels_unscaled.csv")

# Create evaluation dataframes
df_scaled_eval = df_raw.copy()
df_scaled_eval["cluster_id"] = labels_scaled["cluster_id"]

df_unscaled_eval = df_raw.copy()
df_unscaled_eval["cluster_id"] = labels_unscaled["cluster_id"]

# Derive goal boolean for goal_rate
df_scaled_eval["is_goal"] = (df_scaled_eval["outcome"] == "Goal").astype(int)
df_unscaled_eval["is_goal"] = (df_unscaled_eval["outcome"] == "Goal").astype(int)
\n

In [ ]:
# -----------------------------------------------------------
# External Validation (xG vs Actual Goal Rate)
# -----------------------------------------------------------
def cluster_profile(df_eval):
    profile = df_eval.groupby("cluster_id").agg(
        n_shots=("event_id", "count"),
        mean_distance=("distance_to_goal", "mean"),
        mean_angle=("angle_to_goal", "mean"),
        goal_rate=("is_goal", "mean"),
        mean_xg=("statsbomb_xg", "mean")
    ).reset_index()
    return profile

print("=== SCALED CLUSTERS PROFILE ===")
profile_scaled = cluster_profile(df_scaled_eval)
display(profile_scaled)

print("\n=== UNSCALED CLUSTERS PROFILE ===")
profile_unscaled = cluster_profile(df_unscaled_eval)
display(profile_unscaled)

def rank_agreement(profile):
    corr, p = spearmanr(profile["goal_rate"], profile["mean_xg"])
    return corr, p

corr_scaled, _ = rank_agreement(profile_scaled)
corr_unscaled, _ = rank_agreement(profile_unscaled)
print(f"Spearman Rank Correlation (Scaled): {corr_scaled:.3f}")
print(f"Spearman Rank Correlation (Unscaled): {corr_unscaled:.3f}")
\n

In [ ]:
# -----------------------------------------------------------
# Visualizing External Validation
# -----------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Scaled
x = np.arange(len(profile_scaled))
width = 0.35
axes[0].bar(x - width/2, profile_scaled["goal_rate"], width, label="Actual Goal Rate", color="#3498db")
axes[0].bar(x + width/2, profile_scaled["mean_xg"], width, label="Mean xG", color="#e74c3c")
axes[0].set_xticks(x)
axes[0].set_xticklabels(profile_scaled["cluster_id"])
axes[0].set_title("SCALED: Goal Rate vs Mean xG by Cluster")
axes[0].legend()

# Unscaled
x = np.arange(len(profile_unscaled))
axes[1].bar(x - width/2, profile_unscaled["goal_rate"], width, label="Actual Goal Rate", color="#3498db")
axes[1].bar(x + width/2, profile_unscaled["mean_xg"], width, label="Mean xG", color="#e74c3c")
axes[1].set_xticks(x)
axes[1].set_xticklabels(profile_unscaled["cluster_id"])
axes[1].set_title("UNSCALED: Goal Rate vs Mean xG by Cluster")
axes[1].legend()

plt.tight_layout()
plt.show()
\n

In [ ]:
# -----------------------------------------------------------
# Internal Stability Validation (KNN 5-Fold CV)
# -----------------------------------------------------------
def knn_cv(X, y, cv=5, neighbors_grid=[3, 5, 7, 9]):
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)
    results = []
    
    for k in neighbors_grid:
        knn = KNeighborsClassifier(n_neighbors=k, metric="minkowski", p=2)
        scores = cross_val_score(knn, X, y, cv=skf, scoring="accuracy")
        results.append({
            "k_neighbors": k,
            "mean_acc": scores.mean(),
            "std_acc": scores.std(),
            "fold_accs": scores
        })
    return pd.DataFrame(results)

# Prepare X for KNN (we need to apply the exact same one-hot encoding used in modeling)
NUMERIC_FEATURES = ["location_x", "location_y", "distance_to_goal", "angle_to_goal", "n_teammates_in_frame", "n_opponents_in_frame", "keeper_x", "keeper_y"]
CATEGORICAL_FEATURES = ["body_part", "technique", "shot_type"]
BOOL_FLAGS = ["under_pressure", "first_time", "open_goal", "aerial_won"]

X_raw = df_raw[NUMERIC_FEATURES + CATEGORICAL_FEATURES + BOOL_FLAGS].copy()
X_raw["keeper_x"] = X_raw["keeper_x"].fillna(120.0)
X_raw["keeper_y"] = X_raw["keeper_y"].fillna(40.0)
for col in BOOL_FLAGS:
    if col in X_raw.columns:
        X_raw[col] = X_raw[col].fillna(0)
X_encoded = pd.get_dummies(X_raw, columns=CATEGORICAL_FEATURES)
for col in X_encoded.columns:
    if X_encoded[col].dtype == bool:
        X_encoded[col] = X_encoded[col].astype(int)

# 1. Unscaled CV
print("=== KNN CV STABILITY (UNSCALED) ===")
cv_unscaled = knn_cv(X_encoded, df_unscaled_eval["cluster_id"])
display(cv_unscaled[["k_neighbors", "mean_acc", "std_acc"]])

# 2. Scaled CV (Needs scaled features)
from sklearn.preprocessing import RobustScaler
scaler = RobustScaler()
X_scaled_arr = scaler.fit_transform(X_encoded)

print("\n=== KNN CV STABILITY (SCALED) ===")
cv_scaled = knn_cv(X_scaled_arr, df_scaled_eval["cluster_id"])
display(cv_scaled[["k_neighbors", "mean_acc", "std_acc"]])
\n